# Lily 1.5B GRPO Model Benchmark Evaluation — Modal A100
**Evaluates trained Lily-1.5B (GRPO model) using `lm-eval` harness**

Runs multi-fewshot benchmark evaluations (`gsm8k`, `arc_challenge`, `hellaswag`) on `abhinav0231/Lily-1.5B` using Modal A100 GPU with BFloat16 precision and batch size 16.

## Cell 1 — Install Dependencies (Modal %uv optimized)

In [ ]:
# ==============================================================================
# Cell 1 — Dependency Installation (lm-evaluation-harness)
# ==============================================================================
# 1. Uninstall mismatched C++ extensions (torchvision, torchaudio) that break transformers
# 2. Install lm_eval with updated typing_extensions
%uv pip uninstall torchvision torchaudio -q
%uv pip install -U "typing_extensions>=4.12.0" lm_eval[hf] accelerate transformers datasets huggingface_hub wandb -q


## Cell 2 — Hardware Probe & Environment Check

In [ ]:
# ==============================================================================
# Cell 2 — Hardware Probe & Device Verification
# ==============================================================================
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Version    : {torch.version.cuda}")

# Probe GPU compute capability
p = torch.cuda.get_device_properties(0)
print(f"\nGPU Device     : {p.name}")
print(f"VRAM Capacity  : {p.total_memory / 1e9:.1f} GB")
print(f"Compute        : cc={p.major}.{p.minor}")
print(f"BFloat16       : {'Supported' if p.major >= 8 else 'NOT supported'}")

assert torch.cuda.is_available(), "No GPU detected!"
assert p.major >= 8, f"Requires Ampere+ GPU (A100/H100). Got cc={p.major}.{p.minor}"
print("\n✅ Hardware check passed for Modal A100")

## Cell 3 — Evaluation Configuration

In [ ]:
# ==============================================================================
# Cell 3 — Authentication (Hugging Face & W&B) & Evaluation Configuration
# ==============================================================================
import os
try:
    from huggingface_hub import login, get_token
except ImportError:
    from huggingface_hub import login, HfFolder
    get_token = HfFolder.get_token

# Retrieve HF Token from all possible sources (os.environ, Colab Secrets, or cached token)
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")
else:
    print("ℹ️ HF_TOKEN not provided. Proceeding (public datasets/models remain accessible).")

# WandB Authentication
try:
    import wandb
    WANDB_TOKEN = os.environ.get("WANDB_API_KEY", "")
    if WANDB_TOKEN and WANDB_TOKEN != "YOUR_WANDB_KEY_HERE":
        wandb.login(key=WANDB_TOKEN, relogin=True)
        os.environ["WANDB_API_KEY"] = WANDB_TOKEN
        print("✅ Authenticated with Weights & Biases")
    else:
        print("ℹ️ WANDB_API_KEY not found. WandB tracking will operate in offline/disabled mode.")
        os.environ["WANDB_DISABLED"] = "true"
except ImportError:
    print("ℹ️ WandB module not installed. Operating without WandB tracking.")
    os.environ["WANDB_DISABLED"] = "true"

# ------------------------------------------------------------------------------
# Evaluation Configuration Parameters
# ------------------------------------------------------------------------------
HF_USERNAME = "abhinav0231"

# Target Model to Evaluate (Lily-1.5B trained with GRPO from Notebook 04)
MODEL_REPO = f"{HF_USERNAME}/Lily-1.5B"

# Evaluation Benchmarks:
# - gsm8k: Grade-school math reasoning
# - arc_challenge: Multi-hop scientific reasoning
# - hellaswag: Common-sense reasoning & sentence completion
EVAL_TASKS  = "gsm8k,arc_challenge,hellaswag"
NUM_FEWSHOT = 8
BATCH_SIZE  = 16
OUTPUT_DIR  = "/root/lily_1_5b_eval_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nEvaluation Model : {MODEL_REPO}")
print(f"Benchmark Tasks  : {EVAL_TASKS}")
print(f"Few-Shot Setup   : {NUM_FEWSHOT}-shot")
print(f"Batch Size       : {BATCH_SIZE}")
print(f"Output Directory : {OUTPUT_DIR}")


## Cell 4 — Run Benchmark Evaluation (`lm_eval`)

In [ ]:
# ==============================================================================
# Cell 4 — Launch lm-evaluation-harness CLI Subprocess
# ==============================================================================
import subprocess, sys

# Construct CLI command for lm_eval harness
cmd = [
    "lm_eval",
    "--model", "hf",
    "--model_args", f"pretrained={MODEL_REPO},dtype=bfloat16,trust_remote_code=True,attn_implementation=sdpa",
    "--tasks", EVAL_TASKS,
    "--num_fewshot", str(NUM_FEWSHOT),
    "--batch_size", str(BATCH_SIZE),
    "--apply_chat_template",
    "--output_path", OUTPUT_DIR,
    "--device", "cuda:0"
]

print(f"Launching benchmark evaluation suite...")
print(f"CMD: {' '.join(cmd)}\n")

# Stream output in real-time to avoid hidden tracebacks
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="", flush=True)

process.wait()
if process.returncode != 0:
    raise RuntimeError(f"Benchmark evaluation failed with exit code {process.returncode}")
print(f"\n✅ Benchmark evaluation finished successfully!")


## Cell 5 — Display & Parse Evaluation Summary

In [ ]:
# ==============================================================================
# Cell 5 — Parse Evaluation Results JSON & Display Accuracy Breakdown
# ==============================================================================
import glob, json

result_files = glob.glob(f"{OUTPUT_DIR}/**/*.json", recursive=True)
if result_files:
    latest_file = sorted(result_files)[-1]
    print(f"Parsing results file: {latest_file}\n")
    try:
        data = json.load(open(latest_file))
        results = data.get("results", {})
        print("="*50)
        print(f"  BENCHMARK RESULTS — {MODEL_REPO}")
        print("="*50)
        for task, metrics in results.items():
            acc = metrics.get("acc,none", metrics.get("acc", metrics.get("exact_match,none", "N/A")))
            if isinstance(acc, float):
                print(f"  {task:<20} : {acc*100:.2f}%")
            else:
                print(f"  {task:<20} : {acc}")
        print("="*50)
    except Exception as e:
        print(f"Could not parse results JSON: {e}")
else:
    print(f"Results saved in folder: {OUTPUT_DIR}")